In [1]:
import pandas as pd
from pprint import pprint
calendar_df = pd.read_csv('korean_calendar_2024.csv')
weather_df = pd.read_csv('OBS_AWS_TIME_+ridership.csv')
ridership_df = pd.read_csv('seoul_ridership_melted_2024.csv')


In [46]:
# 일시 컬럼을 datetime 형식으로 변환
weather_df['일시'] = pd.to_datetime(weather_df['일시'])

# 날짜와 시간 컬럼 추가
weather_df['date'] = weather_df['일시'].dt.date
weather_df['time'] = weather_df['일시'].dt.time

# 원하는 컬럼 순서로 재배열
weather_df = weather_df[['호선명', '역이름', 'date', 'time', '기온(°C)', '풍향(deg)', '풍속(m/s)', '강수량(mm)', '습도(%)']]
weather_df = weather_df.rename(columns={'호선명': '호선', '역이름': '역명'})
# 결과 확인
weather_df.head()

,호선,역명,date,time,기온(°C),풍향(deg),풍속(m/s),강수량(mm),습도(%)
0,3,대청,2024-01-01,06:00:00,0.7,106.7,0.8,0.0,99.0
1,3,대청,2024-01-01,07:00:00,0.5,92.1,0.9,0.0,98.0
2,3,대청,2024-01-01,08:00:00,0.0,85.1,0.7,0.0,98.0
3,3,대청,2024-01-01,09:00:00,0.5,0.0,0.2,0.0,98.0
4,3,대청,2024-01-01,10:00:00,2.3,0.0,0.3,0.0,97.0


In [ ]:
ridership_df.rename(columns={'날짜': 'date'}, inplace=True)
# 시간 변환 함수 정의
def convert_time_format(time_str):
    # '06시-07시' 형식에서 앞부분만 추출
    hour = int(time_str.split('시-')[0])
    # 형식 변환하여 반환
    return f"{hour:02d}:00:00"

def convert_line_format(line_str):
    # '1호선' 형식에서 숫자만 추출
    return line_str.split('호선')[0]

# 변환 적용
ridership_df['time'] = ridership_df['time'].apply(convert_time_format).dt.time

ridership_df['호선'] = ridership_df['호선'].apply(convert_line_format)

# 결과 확인
print(ridership_df.head())

  호선  역번호   역명        date      time  count
0  1  150  서울역  2024-01-01  06:00:00   1124
1  1  150  서울역  2024-01-01  07:00:00   1142
2  1  150  서울역  2024-01-01  08:00:00   2176
3  1  150  서울역  2024-01-01  09:00:00   3272
4  1  150  서울역  2024-01-01  10:00:00   3451


In [ ]:
calendar_df.head()

  호선  역명        date      time  기온(°C)  풍향(deg)  풍속(m/s)  강수량(mm)  습도(%)
0  3  대청  2024-01-01  06:00:00     0.7    106.7      0.8      0.0   99.0
1  3  대청  2024-01-01  07:00:00     0.5     92.1      0.9      0.0   98.0
2  3  대청  2024-01-01  08:00:00     0.0     85.1      0.7      0.0   98.0
3  3  대청  2024-01-01  09:00:00     0.5      0.0      0.2      0.0   98.0
4  3  대청  2024-01-01  10:00:00     2.3      0.0      0.3      0.0   97.0
  호선  역번호   역명        date      time  count
0  1  150  서울역  2024-01-01  06:00:00   1124
1  1  150  서울역  2024-01-01  07:00:00   1142
2  1  150  서울역  2024-01-01  08:00:00   2176
3  1  150  서울역  2024-01-01  09:00:00   3272
4  1  150  서울역  2024-01-01  10:00:00   3451


In [71]:
len(weather_df['역명'].unique())

27

In [70]:
set(weather_df['역명'].unique()) - set(ridership_df['역명'].unique())

{nan,
 '관악산',
 '구의',
 '남부터미널',
 '독산',
 '동작',
 '마곡나루',
 '보라매공원',
 '북한산보국문',
 '서강대',
 '오목교',
 '온수',
 '응봉',
 '이촌',
 '잠실',
 '회기'}

In [53]:

ridership_df

,호선,역번호,역명,date,time,count
0,1,150,서울역,2024-01-01,06:00:00,1124
1,1,150,서울역,2024-01-01,07:00:00,1142
2,1,150,서울역,2024-01-01,08:00:00,2176
3,1,150,서울역,2024-01-01,09:00:00,3272
4,1,150,서울역,2024-01-01,10:00:00,3451
...,...,...,...,...,...,...
1794685,8,2828,남위례,2024-12-31,19:00:00,836
1794686,8,2828,남위례,2024-12-31,20:00:00,587
1794687,8,2828,남위례,2024-12-31,21:00:00,492
1794688,8,2828,남위례,2024-12-31,22:00:00,469


In [66]:
len(ridership_df['역명'].unique())

241

In [ ]:
weather_df['역명'].unique()

array(['대청', '남부터미널', '명일', '잠실', '마곡나루', '오목교', '쌍문', '석계', '회기', '사가정',
       '보라매공원', '서강대', '신촌', '구의', '북한산보국문', '이촌', '구파발', '독산', '여의나루',
       '서울역', '응봉', '온수', nan, '남태령', '관악산', '영등포시장', '동작'], dtype=object)

In [58]:
# 2) 병합 기준이 될 열 목록 정의
key_cols = ['호선', '역명', 'date', 'time']   # 공통으로 들어 있는 열

# 3) 병합
merged = pd.merge(
    weather_df[weather_df['역명']=='서울역'].head(),
    ridership_df[weather_df['역명']=='서울역'].head(),
    on=key_cols,        # 기준 열
    how='outer'
)
merged

C:\Users\JH\AppData\Local\Temp\ipykernel_2928\1628679618.py:7: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  ridership_df[weather_df['역명']=='서울역'].head(),


IndexingError: Unalignable boolean Series provided as indexer (index of the boolean Series and of the indexed object do not match).

In [19]:
weather_df.loc[weather_df['호선명']=='1',:]['역이름'].unique()

array(['석계', '회기', '독산', '온수(성공회대입구)'], dtype=object)

In [29]:
ridership_df.loc[ridership_df['호선']=='1호선',:]['역명'].unique()

array(['서울역', '시청', '종각', '종로3가', '종로5가', '동대문', '신설동', '제기동',
       '청량리(서울시립대입구)', '동묘앞'], dtype=object)

In [5]:
%conda install lxml

Channels:
 - defaults
Platform: win-64
Solving environment: ...working... done

## Package Plan ##

  environment location: c:\Users\user\anaconda3\envs\WEBSC

  added / updated specs:
    - lxml


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    libiconv-1.16              |       h2bbff1b_3         685 KB
    libxml2-2.13.7             |       h866ff63_0         2.9 MB
    libxslt-1.1.41             |       h0739af5_0         467 KB
    lxml-5.3.0                 |  py313h395c83e_1         1.1 MB
    ------------------------------------------------------------
                                           Total:         5.1 MB

The following NEW packages will be INSTALLED:

  libiconv           pkgs/main/win-64::libiconv-1.16-h2bbff1b_3 
  libxml2            pkgs/main/win-64::libxml2-2.13.7-h866ff63_0 
  libxslt            pkgs/main/win-64::libxslt-1.1.41-h0739af5_0 
  lxml               pk

# 날씨 데이터 수집 크롤링

In [ ]:
"""
pip install requests beautifulsoup4 lxml pandas tqdm
"""

import datetime as dt
import re
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm

# ---------------- 공통 상수 ----------------
BASE_URL = "https://www.weather.go.kr/w/observation/land/city-obs.do"
COMMON_QS = {
    "auto_man": "m",
    "stn": "0",
    "dtm": "",
    "type": "t99",
    "reg": "109",  # 서울·인천·경기도
}
COLS = [
    "date", "time",
    "일기", "시정", "운량", "중하운량",
    "현재기온", "이슬점온도", "체감온도",
    "일강수", "적설", "습도", "풍향", "풍속", "해면기압",
]

# ---------------- 유틸 함수 ----------------
def build_params(ts: dt.datetime) -> dict:
    tm_raw = ts.strftime("%Y.%m.%d.%H:00")
    return COMMON_QS | {"tm": tm_raw}

def parse_row(tr, ts):
    if tr is None:
        return None

    tds = tr.find_all("td")
    cells = []
    for td in tds:
        if td.script:  # 풍속 셀
            m = re.search(r"writeWindSpeed\('([\d.]+)'", td.script.string)
            cells.append(float(m.group(1)) if m else None)
        else:
            txt = td.get_text(strip=True).replace("−", "-")
            cells.append(
                float(txt) if re.fullmatch(r"-?\d+(\.\d+)?", txt) else (txt or None)
            )

    # 적설(tds 인덱스 8)이 없어서 cells가 12개라면, 0.0을 끼워넣기
    if len(cells) == 12:
        cells.insert(8, 0.0)

    # 그래도 13개가 안 되면 건너뛰기
    if len(cells) != 13:
        return None

    return {
        "date": ts.strftime("%Y-%m-%d"),
        "time": ts.strftime("%H:%M"),
        **dict(zip(COLS[2:], cells))
    }


def fetch_one_hour(ts, retries=3, backoff=3):
    """ts 시각의 서울 관측 데이터를 재시도하며 가져옵니다."""
    for attempt in range(1, retries+1):
        try:
            resp = requests.get(BASE_URL, params=build_params(ts), timeout=8)
            resp.raise_for_status()
            soup = BeautifulSoup(resp.text, "lxml")
            link = soup.select_one('a[href*="stn=108"]')  # 서울 stn=108
            # print(f"[{ts}] 서울 관측 데이터: {link.find_parent('tr')}")
            if not link:
                print(f"[{ts}] ❌ 서울 관측 데이터 없음")
                return None
            return parse_row(link.find_parent('tr'), ts)
        except requests.RequestException as e:
            if attempt < retries:
                time.sleep(backoff * attempt)
            else:
                print(f"[{ts}] 요청 실패({attempt}회): {e}")
    return None

# ---------------- 메인 크롤러 ----------------
def crawl():
    start = dt.datetime(2024, 1, 1, 6)
    end   = dt.datetime(2024, 12, 31, 23)

    # 수집할 ts 목록 (06~23시만)
    timestamps, ts = [], start
    while ts <= end:
        timestamps.append(ts)
        ts += dt.timedelta(hours=1)
        if ts.hour == 0:
            ts += dt.timedelta(hours=6)
    print(f"수집할 데이터 수: {len(timestamps)}")
    rows = []
    pbar = tqdm(total=len(timestamps), desc="Downloading")
    for ts in timestamps:
        row = fetch_one_hour(ts)
        if row:
            rows.append(row)
        pbar.update(1)
    pbar.close()

    if not rows:
        raise RuntimeError("데이터를 한 건도 수집하지 못했습니다.")
    return pd.DataFrame(rows, columns=COLS)

# ---------------- 실행 ----------------
if __name__ == "__main__":
    df = crawl()                   # 약 6 570 행 (365×18)
    df.to_csv("kma_city_obs_2024_seoul.csv", index=False, encoding="utf-8-sig")
    print(df.head())


[datetime.datetime(2024, 1, 1, 6, 0), datetime.datetime(2024, 1, 1, 7, 0), datetime.datetime(2024, 1, 1, 8, 0), datetime.datetime(2024, 1, 1, 9, 0), datetime.datetime(2024, 1, 1, 10, 0), datetime.datetime(2024, 1, 1, 11, 0), datetime.datetime(2024, 1, 1, 12, 0), datetime.datetime(2024, 1, 1, 13, 0), datetime.datetime(2024, 1, 1, 14, 0), datetime.datetime(2024, 1, 1, 15, 0), datetime.datetime(2024, 1, 1, 16, 0), datetime.datetime(2024, 1, 1, 17, 0), datetime.datetime(2024, 1, 1, 18, 0), datetime.datetime(2024, 1, 1, 19, 0), datetime.datetime(2024, 1, 1, 20, 0), datetime.datetime(2024, 1, 1, 21, 0), datetime.datetime(2024, 1, 1, 22, 0), datetime.datetime(2024, 1, 1, 23, 0), datetime.datetime(2024, 1, 2, 6, 0), datetime.datetime(2024, 1, 2, 7, 0), datetime.datetime(2024, 1, 2, 8, 0), datetime.datetime(2024, 1, 2, 9, 0), datetime.datetime(2024, 1, 2, 10, 0), datetime.datetime(2024, 1, 2, 11, 0), datetime.datetime(2024, 1, 2, 12, 0), datetime.datetime(2024, 1, 2, 13, 0), datetime.datetime(2

AttributeError: 'NoneType' object has no attribute 'head'

In [19]:
"""
pip install requests beautifulsoup4 lxml pandas tqdm
"""

import datetime as dt
import re
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm

# ---------------- 공통 상수 ----------------
BASE_URL = "https://www.weather.go.kr/w/observation/land/city-obs.do"
COMMON_QS = {
    "auto_man": "m",
    "stn": "0",
    "dtm": "",
    "type": "t99",
    "reg": "109",  # 서울·인천·경기도
}
COLS = [
    "date", "time",
    "일기", "시정", "운량", "중하운량",
    "현재기온", "이슬점온도", "체감온도",
    "일강수", "적설", "습도", "풍향", "풍속", "해면기압",
]

# ---------------- 유틸 함수 ----------------
def build_params(ts: dt.datetime) -> dict:
    tm_raw = ts.strftime("%Y.%m.%d.%H:00")
    return COMMON_QS | {"tm": tm_raw}

def parse_row(tr, ts):
    if tr is None:
        return None

    tds = tr.find_all("td")
    cells = []
    for td in tds:
        if td.script:  # 풍속 셀
            m = re.search(r"writeWindSpeed\('([\d.]+)'", td.script.string)
            cells.append(float(m.group(1)) if m else None)
        else:
            txt = td.get_text(strip=True).replace("−", "-")
            cells.append(
                float(txt) if re.fullmatch(r"-?\d+(\.\d+)?", txt) else (txt or None)
            )

    # 적설(tds 인덱스 8)이 없어서 cells가 12개라면, 0.0을 끼워넣기
    if len(cells) == 12:
        cells.insert(8, 0.0)

    # 그래도 13개가 안 되면 건너뛰기
    if len(cells) != 13:
        return None

    return {
        "date": ts.strftime("%Y-%m-%d"),
        "time": ts.strftime("%H:%M"),
        **dict(zip(COLS[2:], cells))
    }


def fetch_one_hour(ts, retries=3, backoff=3):
    """ts 시각의 서울 관측 데이터를 재시도하며 가져옵니다."""
    for attempt in range(1, retries+1):
        try:
            resp = requests.get(BASE_URL, params=build_params(ts), timeout=8)
            resp.raise_for_status()
            soup = BeautifulSoup(resp.text, "lxml")
            link = soup.select_one('a[href*="stn=108"]')  # 서울 stn=108
            # print(f"[{ts}] 서울 관측 데이터: {link.find_parent('tr')}")
            if not link:
                print(f"[{ts}] ❌ 서울 관측 데이터 없음")
                return None
            return parse_row(link.find_parent('tr'), ts)
        except requests.RequestException as e:
            if attempt < retries:
                time.sleep(backoff * attempt)
            else:
                print(f"[{ts}] 요청 실패({attempt}회): {e}")
    return None

# ---------------- 누락 시각 계산 ----------------
def get_missing_timestamps(existing_csv):
    df = pd.read_csv(existing_csv)
    df["datetime"] = pd.to_datetime(df["date"] + " " + df["time"])
    start = dt.datetime(2024,1,1,6)
    end   = dt.datetime(2024,12,31,23)
    expected = []
    ts = start
    while ts <= end:
        expected.append(ts)
        ts += dt.timedelta(hours=1)
        if ts.hour == 0:
            ts += dt.timedelta(hours=6)
    actual = set(df["datetime"].dt.to_pydatetime())
    missing = [t for t in expected if t not in actual]
    return missing, df.drop(columns="datetime")

# ---------------- 재수집 & 병합 ----------------
def refill_and_merge(existing_csv, output_csv):
    missing, df_exist = get_missing_timestamps(existing_csv)
# (b) 누락된 날짜만 보기
    missing_dates = sorted({ts.date() for ts in missing})
    print("\n누락된 날짜:")
    for d in missing_dates:
        print(d)
    
    print(f"누락된 시각 {len(missing)}개 재수집 시작…")
    
    new_rows = []
    for ts in tqdm(missing, desc="Retry missing"):
        row = fetch_one_hour(ts)
        if row:
            new_rows.append(row)
    
    if not new_rows:
        print("새로 받은 데이터가 없습니다.")
        df_exist.to_csv(output_csv, index=False, encoding="utf-8-sig")
        return df_exist
    
    df_new = pd.DataFrame(new_rows, columns=COLS)
    print(f"새로 받은 데이터 {len(df_new)}개")

    df_all = pd.concat([df_exist, df_new], ignore_index=True)
    df_all.drop_duplicates(subset=["date","time"], keep="first", inplace=True)
    df_all.sort_values(["date","time"], inplace=True)
    df_all.to_csv(output_csv, index=False, encoding="utf-8-sig")
    print(f"병합 완료: {output_csv}")
    return df_all

# ---------------- 실행 예 ----------------
if __name__ == "__main__":
    # 기존에 저장된 서울 데이터 파일명
    EXISTING_CSV = "kma_city_obs_2024_seoul.csv"
    OUTPUT_CSV   = "kma_city_obs_2024_seoul.csv"
    merged_df = refill_and_merge(EXISTING_CSV, OUTPUT_CSV)
    print("\n최종 데이터 개수:", len(merged_df))


C:\Users\user\AppData\Local\Temp\ipykernel_16664\951938036.py:99: FutureWarning: The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result
  actual = set(df["datetime"].dt.to_pydatetime())



누락된 날짜:
2024-05-02
2024-05-03
2024-05-04
2024-05-05
2024-05-06
2024-05-07
2024-05-08
2024-05-09
2024-05-10
2024-05-11
2024-05-12
2024-05-13
2024-05-14
2024-05-15
2024-05-16
2024-05-17
2024-05-18
2024-05-19
2024-05-20
2024-05-21
2024-05-22
2024-05-23
2024-05-24
2024-05-25
2024-05-26
2024-05-27
2024-05-28
2024-05-29
2024-05-30
2024-05-31
2024-06-01
2024-06-02
2024-06-03
2024-06-04
2024-06-05
2024-06-06
2024-06-07
2024-06-08
2024-06-09
2024-06-10
2024-06-11
2024-06-12
2024-06-13
2024-06-14
2024-06-15
2024-06-16
2024-06-17
2024-06-18
2024-06-19
2024-06-20
2024-06-21
2024-06-22
2024-06-23
2024-06-24
2024-06-25
2024-06-26
2024-06-27
2024-06-28
2024-06-29
2024-06-30
2024-07-01
2024-07-02
2024-07-03
2024-07-04
2024-07-05
2024-07-06
2024-07-07
2024-07-08
2024-07-09
2024-07-10
2024-07-11
2024-07-12
2024-07-13
2024-07-14
2024-07-15
2024-07-16
2024-07-17
2024-07-18
2024-07-19
2024-07-20
2024-07-21
2024-07-22
2024-07-23
2024-07-24
2024-07-25
2024-07-26
2024-07-27
2024-07-28
2024-07-29
2024-07-30
2

Retry missing: 100%|██████████| 2736/2736 [06:19<00:00,  7.21it/s]

새로 받은 데이터 2736개
병합 완료: kma_city_obs_2024_seoul.csv

최종 데이터 개수: 6588


In [2]:
import pandas as pd
import numpy as np
import re

# -------- 1) CSV 불러오기 --------
df_ridership = pd.read_csv("seoul_ridership_melted_2024.csv")
df_weather   = pd.read_csv("kma_city_obs_2024_seoul.csv")
df_calendar  = pd.read_csv("korean_calendar_2024.csv")

# 데이터 셋 병합

In [8]:
import pandas as pd
import numpy as np
import re

# -------- 1) CSV 불러오기 --------
df_ridership = pd.read_csv("seoul_ridership_melted_2024.csv")
df_weather   = pd.read_csv("kma_city_obs_2024_seoul.csv")
df_calendar  = pd.read_csv("korean_calendar_2024.csv")

# -------- 2) 시간·날짜 정규화 --------
df_ridership["time"] = df_ridership["time"].str[:2] + ":00"

# --------- 3) 결측치 처리 --------

# (1) forward fill 할 컬럼
forward_fill_cols = ["일기", "운량", "중하운량", "체감온도", "풍향", "풍속"]
df_weather[forward_fill_cols] = df_weather[forward_fill_cols].ffill()

# (2) 0.0 채울 컬럼
zero_fill_cols = ["적설", "일강수"]
df_weather[zero_fill_cols] = df_weather[zero_fill_cols].fillna(0.0)

# -------- 3) 숫자 컬럼을 float 로 깨끗이 변환 --------
num_cols = ["시정","운량","중하운량","현재기온","이슬점온도","체감온도",
            "일강수","적설","습도","풍속","해면기압"]

def to_float(x):
    """
    '20 이상', '3.5', '-', np.nan 등에서 숫자만 추출해 float 반환.
    숫자 없으면 np.nan
    """
    if pd.isna(x):
        return np.nan
    m = re.search(r"-?\d+(?:\.\d+)?", str(x))
    return float(m.group()) if m else 0.0

df_weather[num_cols] = df_weather[num_cols].applymap(to_float)

df_merged = pd.merge(
    df_ridership, df_weather, how="left", on=["date", "time"]
)

df_merged = pd.merge(
    df_merged, df_calendar, how="left", on=["date"]
)

df_merged.to_csv("seoul_ridership_weather_calendar_2024.csv", index=False, encoding="utf-8")

C:\Users\user\AppData\Local\Temp\ipykernel_10548\226078602.py:37: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_weather[num_cols] = df_weather[num_cols].applymap(to_float)


In [1]:
dataset_df = pd.read_csv('seoul_ridership_weather_calendar_2024.csv')
dataset_df = dataset_df.sample(100)

NameError: name 'pd' is not defined

In [3]:
import pandas as pd
location_df = pd.read_csv('서울교통공사_1_8호선 역사 좌표(위경도) 정보_20241031.csv', encoding='cp949')
ridership_df = pd.read_csv('seoul_ridership_melted_2024.csv', encoding='utf-8')
# print(location_df['고유역번호(외부역코드)'].unique())
print(set(location_df['고유역번호(외부역코드)'].unique()) - set(ridership_df['station_code'].unique()))

{np.int64(200), np.int64(2649), np.int64(321), np.int64(2615)}


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import timedelta


url = 'https://www.weather.go.kr/w/wnuri-fct2021/main/digital-forecast.do?code=1114055000&unit=m%2Fs&hr1=Y&ts=&lat=37.5513049702718&lon=126.988230622132'
response = requests.get(url)
# 응답 저장
with open('weather_forecast.html', 'w', encoding='utf-8') as file:
    file.write(response.text)
soup = BeautifulSoup(response.text, 'html.parser')

# -------- 유틸 --------
num = lambda s: re.search(r"-?\d+", s).group() if re.search(r"-?\d+", s) else None

def parse_rain(text: str) -> float:
    """ '~1mm' → 0.5, '-' → 0, '3mm' → 3  """
    if "~1" in text:           return 0.5
    if "-"  in text or text.strip() == "": return 0.0
    return float(num(text))

def combine_dt(date_str, time_str):
    """24:00 → 다음날 00:00으로 보정해 datetime 반환"""
    d = pd.to_datetime(date_str)
    h, m = map(int, time_str.split(":"))
    if h == 24:
        d += timedelta(days=1); h = 0
    return d + timedelta(hours=h, minutes=m)

# -------- 본체 --------
records = []
for ul in soup.select("ul.item.s-item, ul.item.vs-item"):  # 1시간·3시간 간격 모두
    li = ul.select("li")

    record = {
        "일자"     : ul["data-date"],
        "시간"     : ul["data-time"],
        # ※ li[1] 안 <span class="wic"> 가 실제 상태
        "일기"     : li[1].select_one(".wic").get_text(strip=True),
        "온도"     : num(li[2].get_text()),
        "체감온도" : num(li[3].get_text()),
        "강수량"   : parse_rain(li[4].get_text()),
        "강수확률" : num(li[6].get_text()) or 0,
        # 풍향: title="남동풍" → '남동풍' / 풍속: 마지막 <span> 안의 숫자
        "풍향"     : li[7].select_one("span[title]")["title"][:-1],
        "풍속"     : num(li[7].get_text()),
        "습도"     : num(li[8].get_text()),
    }
    records.append(record)

df = pd.DataFrame(records)

# 숫자열 변환
for c in ["온도","체감온도","강수량","강수확률","풍속","습도"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# datetime 열
df["datetime"] = df.apply(lambda r: combine_dt(r["일자"], r["시간"]), axis=1)




,일자,시간,일기,온도,체감온도,강수량,강수확률,풍향,풍속,습도,datetime
0,2025-04-22,04:00,흐림,19,19,0.0,0,동,2,80,2025-04-22 04:00:00
1,2025-04-22,05:00,흐림,19,19,0.0,0,동,2,85,2025-04-22 05:00:00
2,2025-04-22,06:00,흐림,17,17,0.0,0,동,3,85,2025-04-22 06:00:00
3,2025-04-22,07:00,약한비,17,17,0.5,0,동,4,85,2025-04-22 07:00:00
4,2025-04-22,08:00,약한비,17,17,2.0,0,남동,5,85,2025-04-22 08:00:00
...,...,...,...,...,...,...,...,...,...,...,...
64,2025-04-24,20:00,맑음,15,15,0.0,0,남서,1,45,2025-04-24 20:00:00
65,2025-04-24,21:00,맑음,13,13,0.0,0,북서,2,40,2025-04-24 21:00:00
66,2025-04-24,22:00,맑음,13,13,0.0,0,북서,2,40,2025-04-24 22:00:00
67,2025-04-24,23:00,맑음,11,11,0.0,0,북서,2,40,2025-04-24 23:00:00


In [ ]:
"""
pip install requests beautifulsoup4 lxml pandas tqdm
"""

import datetime as dt
import re
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup

# ---------------- 공통 상수 ----------------
BASE_URL = "https://www.weather.go.kr/w/observation/land/city-obs.do"
COMMON_QS = {
    "auto_man": "m",
    "stn": "0",
    "dtm": "",
    "type": "t99",
    "reg": "109",  # 서울·인천·경기도
}
COLS = [
    "date", "time",
    "일기", "시정", "운량", "중하운량",
    "현재기온", "이슬점온도", "체감온도",
    "일강수", "적설", "습도", "풍향", "풍속", "해면기압",
]

# ---------------- 유틸 함수 ----------------
def build_params(ts: dt.datetime) -> dict:
    tm_raw = ts.strftime("%Y.%m.%d.%H:00")
    return COMMON_QS | {"tm": tm_raw}

def parse_row(tr, ts):
    if tr is None:
        return None

    tds = tr.find_all("td")
    cells = []
    for td in tds:
        if td.script:  # 풍속 셀
            m = re.search(r"writeWindSpeed\('([\d.]+)'", td.script.string)
            cells.append(float(m.group(1)) if m else None)
        else:
            txt = td.get_text(strip=True).replace("−", "-")
            cells.append(
                float(txt) if re.fullmatch(r"-?\d+(\.\d+)?", txt) else (txt or None)
            )

    # 적설(tds 인덱스 8)이 없어서 cells가 12개라면, 0.0을 끼워넣기
    if len(cells) == 12:
        cells.insert(8, 0.0)

    # 그래도 13개가 안 되면 건너뛰기
    if len(cells) != 13:
        return None

    return {
        "date": ts.strftime("%Y-%m-%d"),
        "time": ts.strftime("%H:%M"),
        **dict(zip(COLS[2:], cells))
    }


def fetch_one_hour(retries=3, backoff=3):
    for attempt in range(1, retries+1):
        try:
            resp = requests.get(BASE_URL, timeout=8)
            resp.raise_for_status()
            soup = BeautifulSoup(resp.text, "lxml")
            full_text = soup.get_text(" ", strip=True)   # <br>, 공백 등을 한 칸으로 치환

            # 2) yyyy.mm.dd.hh:mm 형태에 매칭되는 정규식
            dt_pattern = re.compile(r"\b\d{4}\.\d{2}\.\d{2}\.\d{2}:\d{2}\b")

            # 3) 첫 번째(or 모든) 매칭 가져오기
            matches = dt_pattern.findall(full_text)
            # 4) datetime 객체로 변환
            ts = dt.datetime.strptime(matches[0], "%Y.%m.%d.%H:%M") if matches else None
            link = soup.select_one('a[href*="stn=108"]')  # 서울 stn=108
            if not link:
                print(f"[{ts}] ❌ 서울 관측 데이터 없음")
                return None
            return parse_row(link.find_parent('tr'), ts)
        except requests.RequestException as e:
            if attempt < retries:
                time.sleep(backoff * attempt)
            else:
                print(f"[{ts}] 요청 실패({attempt}회): {e}")
    return None

# ---------------- 메인 크롤러 ----------------
def crawl():
    row = fetch_one_hour()
    return pd.DataFrame([row], columns=COLS) if row else pd.DataFrame(columns=COLS)

# ---------------- 실행 ----------------
if __name__ == "__main__":
    df = crawl()                   # 1개행 데이터 프레임
    print(df.head())


2025.04.22.02:00
2025-04-22 02:00:00
         date   time        일기    시정    운량  중하운량  현재기온  이슬점온도  체감온도  일강수  \
0  2025-04-22  02:00  약한 비 연속적  10.3  10.0   9.0  20.1   15.9  20.1  0.0   

     적설    습도 풍향   풍속    해면기압  
0  None  77.0  서  1.0  1009.0  


In [10]:
import pandas as pd
import numpy as np
import re
import joblib

import datetime as dt
import re
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup

# ---------------- 공통 상수 ----------------
BASE_URL = "https://www.weather.go.kr/w/observation/land/city-obs.do"
COMMON_QS = {
    "auto_man": "m",
    "stn": "0",
    "dtm": "",
    "type": "t99",
    "reg": "109",  # 서울·인천·경기도
}
COLS = [
    "date", "time",
    "일기", "시정", "운량", "중하운량",
    "현재기온", "이슬점온도", "체감온도",
    "일강수", "적설", "습도", "풍향", "풍속", "해면기압",
]

# ---------------- 유틸 함수 ----------------
def build_params(ts: dt.datetime) -> dict:
    tm_raw = ts.strftime("%Y.%m.%d.%H:00")
    return COMMON_QS | {"tm": tm_raw}

def parse_row(tr, ts):
    if tr is None:
        return None

    tds = tr.find_all("td")
    cells = []
    for td in tds:
        if td.script:  # 풍속 셀
            m = re.search(r"writeWindSpeed\('([\d.]+)'", td.script.string)
            cells.append(float(m.group(1)) if m else None)
        else:
            txt = td.get_text(strip=True).replace("−", "-")
            cells.append(
                float(txt) if re.fullmatch(r"-?\d+(\.\d+)?", txt) else (txt or None)
            )

    # 적설(tds 인덱스 8)이 없어서 cells가 12개라면, 0.0을 끼워넣기
    if len(cells) == 12:
        cells.insert(8, 0.0)

    # 그래도 13개가 안 되면 건너뛰기
    if len(cells) != 13:
        return None

    return {
        "date": ts.strftime("%Y-%m-%d"),
        "time": ts.strftime("%H:%M"),
        **dict(zip(COLS[2:], cells))
    }


def fetch_one_hour(retries=3, backoff=3):
    for attempt in range(1, retries+1):
        try:
            resp = requests.get(BASE_URL, timeout=8)
            resp.raise_for_status()
            soup = BeautifulSoup(resp.text, "lxml")
            full_text = soup.get_text(" ", strip=True)   # <br>, 공백 등을 한 칸으로 치환

            # 2) yyyy.mm.dd.hh:mm 형태에 매칭되는 정규식
            dt_pattern = re.compile(r"\b\d{4}\.\d{2}\.\d{2}\.\d{2}:\d{2}\b")

            # 3) 첫 번째(or 모든) 매칭 가져오기
            matches = dt_pattern.findall(full_text)
            # 4) datetime 객체로 변환
            ts = dt.datetime.strptime(matches[0], "%Y.%m.%d.%H:%M") if matches else None
            link = soup.select_one('a[href*="stn=108"]')  # 서울 stn=108
            if not link:
                print(f"[{ts}] ❌ 서울 관측 데이터 없음")
                return None
            return parse_row(link.find_parent('tr'), ts)
        except requests.RequestException as e:
            if attempt < retries:
                time.sleep(backoff * attempt)
            else:
                print(f"[{ts}] 요청 실패({attempt}회): {e}")
    return None

# ---------------- 메인 크롤러 ----------------
def crawl():
    row = fetch_one_hour()
    return pd.DataFrame([row], columns=COLS) if row else pd.DataFrame(columns=COLS)

# ---------------- 실행 ----------------
    

def load_congestion_model():
    """
    저장된 XGBoost 모델을 로드
    """
    model = joblib.load('subway_xgb_best.pkl')
    return model

def predict_all_stations_congestion(model, current_weather, all_stations):
    """
    현재 날씨 데이터를 기반으로 모든 역의 혼잡도 예측
    """
    # 현재 시간과 요일 정보
    now = dt.datetime.now()
    current_weekday = now.weekday()  # 월=0, 일=6
    
    # 2025년 한국 공휴일 정보
    holidays_2025 = [
        "2025-01-01",  # 신정
        "2025-01-27",  # 설날 연휴
        "2025-01-28",  # 설날
        "2025-01-29",  # 설날 연휴
        "2025-01-30",  # 설날 연휴
        "2025-03-01",  # 삼일절
        "2025-03-03",  # 대체 공휴일
        "2025-05-05",  # 어린이날
        "2025-05-06",  # 대체 공휴일
        "2025-06-03",  # 현충일
        "2025-06-06",  # 대체 공휴일
        "2025-08-15",  # 광복절
        "2025-10-03",  # 개천절
        "2025-10-05",  # 추석 연휴
        "2025-10-06",  # 추석
        "2025-10-07",  # 추석 연휴
        "2025-10-08",  # 추석 연휴
        "2025-10-09",  # 한글날
        "2025-12-25",  # 성탄절
    ]
    
    # 현재 날짜 문자열 형식으로 가져오기 (YYYY-MM-DD)
    current_date = now.strftime("%Y-%m-%d")
    
    # 휴일 여부 확인 (주말 또는 공휴일)
    is_holiday = 1 if current_date in holidays_2025 else 0
    
    # 결과를 저장할 데이터프레임
    results = []
    
    # 각 역별로 예측
    for station in all_stations:
        # 모든 필요한 특성 구성
        input_features = {
            'date' : current_weather.get('date', now.strftime("%Y-%m-%d")),
            'line': station['line'],
            'station_code': station['station_code'],
            'station_name': station['station_name'],
            'time': current_weather.get('time', now.strftime("%H:%M")),
            '일기': current_weather.get('일기', '맑음'),
            '시정': current_weather.get('시정', 10),
            '운량': current_weather.get('운량', 0),
            '중하운량': current_weather.get('중하운량', 0),
            '현재기온': current_weather.get('현재기온', 20),
            '이슬점온도': current_weather.get('이슬점온도', 10),
            '체감온도': current_weather.get('체감온도', 20),
            '일강수': current_weather.get('일강수', 0),
            '적설': current_weather.get('적설', 0),
            '습도': current_weather.get('습도', 60),
            '풍향': current_weather.get('풍향', '남남서'),
            '풍속': current_weather.get('풍속', 2),
            '해면기압': current_weather.get('해면기압', 1013),
            'weekday': current_weekday,
            'is_holiday': is_holiday,
        }
        
        # 데이터프레임 생성
        X_pred = pd.DataFrame([input_features])
        
        drop_cols = ['date', 'time', 'station_name']
        X_pred = X_pred.drop(columns=drop_cols)
        # 승하차 인원 예측
        return model.predict(X_pred)[0]
    
    return pd.DataFrame(results)



model = load_congestion_model()
df = crawl()                   # 1개행 데이터 프레임
predict_all_stations_congestion(model, df, [{'line': '1호선', 'station_code': '101', 'station_name': '서울역'}])


ValueError: columns are missing: {'day', 'hour', 'month'}